# Class Exercises
1. Dataset is from: https://www.kaggle.com/datasets/imdevskp/nobel-prize

2. Use SQL or Spark SQL to create a schema under data catalog and a volume under that schemas.

Sample:
```sql
CREATE SCHEMA
CREATE VOLUME
CREATE CATALOG
```
3. Upload data into the volume

4. Read in data into spark

5. Count how many Swedish Nobel Prizes medals there are

6. Short EDA of your choice - preferably with some graphs also

# Load Nobel Prize Dataset

### Checking what we have first

In [0]:
%sql
-- Check my current catalog first
-- In the upper right, it shows "workspace", but I can still create a catalog in other places like "data
SELECT current_catalog();

In [0]:
%sql
-- Show the catalogs that I currently have
SHOW CATALOGS;



In [0]:
-- Show the current schemas I have in the data catalog
SHOW SCHEMAS IN data;

### Create the Schemas, Volumes and Catalogs

In [0]:
%sql
-- Create the Schema
CREATE SCHEMA IF NOT EXISTS data.nobel_prize;

In [0]:
-- Show the schemas in the data catalog
SHOW SCHEMAS in data 

In [0]:
%sql
-- Create the Volume
CREATE VOLUME IF NOT EXISTS data.nobel_prize.raw_data;

In [0]:
%sql
-- Show the created volumes
SHOW VOLUMES IN data.nobel_prize;

In [0]:
LIST "/Volumes/data/nobel_prize/raw_data"

### Uploading the dataset in the catalog
Then go to the catalog in the side bar after creating the Schema, volumes. Upload the csv file into that "raw_data" catalog.

### Load the dataset using spark sql

In [0]:
%python
# Load data using spark sql
DATA_PATH = "/Volumes/data/nobel_prize/raw_data/complete.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

display(df)

In [0]:
%python
display(df.limit(3))

In [0]:
%python
# print the schema
df.printSchema()

In [0]:
%python
# count the rows
df.count()

In [0]:
%python
# print the columns
df.columns

## EDA of the Nobel Prize Dataset

In [0]:
%python

swedish_nobel_prize_df = df.filter(df.birth_countryNow == "Sweden")

display(swedish_nobel_prize_df)

In [0]:
%python
# Sweden has won 29 awards
swedish_nobel_prize_df.count()

In [0]:
%python
# Import this dataset that shows number of Swedish Nobe Prize Awards
from pyspark.sql.functions import col

swedish_count = df.filter(col("birth_countryNow") == "Sweden").count()

print(f"Number of Swedish Nobel Prize medals: {swedish_count}")

In [0]:
%python

from pyspark.sql.functions import col
import matplotlib.pyplot as plt

# Count Swedish Nobel Prize medals
swedish_count = df.filter(col("birth_countryNow") == "Sweden").count()

# Create simple bar chart
plt.figure(figsize=(6, 4))
plt.bar(["Sweden"], [swedish_count])

plt.title("Number of Swedish Nobel Prize Medals")
plt.xlabel("Country")
plt.ylabel("Number of medals")

plt.show()

In [0]:
%python

from pyspark.sql.functions import col, count
import matplotlib.pyplot as plt

# Nordic countries to compare
nordic_countries = ["Sweden", "Norway", "Denmark", "Finland", "Iceland"]

# Count Nobel Prize medals by current birth country
nordic_counts_df = (
    df.filter(col("birth_countryNow").isin(nordic_countries))
      .groupBy("birth_countryNow")
      .agg(count("*").alias("medal_count"))
      .orderBy(col("medal_count").desc())
)

display(nordic_counts_df)

In [0]:
%python

# Convert Spark DataFrame to Pandas for matplotlib
nordic_counts_pd = nordic_counts_df.toPandas()

# Create bar chart
plt.figure(figsize=(8, 5))
plt.bar(
    nordic_counts_pd["birth_countryNow"],
    nordic_counts_pd["medal_count"]
)

plt.title("Nobel Prize Medals by Nordic Country")
plt.xlabel("Country")
plt.ylabel("Number of medals")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [0]:
%python
# I want to check these 2 columns birth_country, 'birth_countryNow'

display(
    df.select("birth_country", "birth_countryNow")
      .distinct()
      .orderBy("birth_countryNow", "birth_country")
)

#### COunt total number of Nobel Prize Records

In [0]:
%python

total_records = df.count()

print(f"Total number of Nobel Prize records: {total_records}")

#### Count of Nobel Prize Awards Per Category

In [0]:
%python

from pyspark.sql.functions import count, col

category_counts_df = (
    df.groupBy("category")
      .agg(count("*").alias("medal_count"))
      .orderBy(col("medal_count").desc())
)

display(category_counts_df)

#### Nobel Prize Records by Gender

In [0]:
%python

gender_counts_df = (
    df.groupBy("gender")
      .agg(count("*").alias("medal_count"))
      .orderBy(col("medal_count").desc())
)

display(gender_counts_df)